<a href="https://colab.research.google.com/github/hyunkyung31/coronary-ai-ml-dl/blob/main/yuri/0901_COCA_SegResNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
START_EPOCH = 21
MAX_EPOCHS = 50
VAL_INTERVAL = 5

print(
    f"{START_EPOCH} epoch부터 "
    f"{MAX_EPOCHS} epoch까지 학습"
)

print("현재 최고 Dice:", best_metric)
print("현재 최고 epoch:", best_metric_epoch)
print("기존 기록:", len(training_history))

21 epoch부터 50 epoch까지 학습
현재 최고 Dice: 0.2556348145008087
현재 최고 epoch: 20
기존 기록: 20


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=list_data_collate,
    pin_memory=True,
    persistent_workers=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False,
)


print("학습 batch:", len(train_loader))
print("검증 batch:", len(val_loader))

학습 batch: 347
검증 batch: 43


In [ ]:
last_checkpoint_path = (
    MODEL_DIR / "segresnet_last.pth"
)

checkpoint = torch.load(
    last_checkpoint_path,
    map_location=device,
    weights_only=False,
)


model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

scaler.load_state_dict(
    checkpoint["scaler_state_dict"]
)


last_completed_epoch = int(
    checkpoint["epoch"]
)

START_EPOCH = last_completed_epoch + 1
MAX_EPOCHS = 50


best_metric = float(
    checkpoint["best_metric"]
)

best_metric_epoch = int(
    checkpoint["best_metric_epoch"]
)


history_path = (
    MODEL_DIR
    / "segresnet_training_history.csv"
)

history_dataframe = pd.read_csv(
    history_path
)

# 마지막 정상 저장 epoch까지만 유지
history_dataframe = history_dataframe[
    history_dataframe["epoch"]
    <= last_completed_epoch
].copy()

training_history = (
    history_dataframe.to_dict("records")
)


print("마지막 정상 epoch:", last_completed_epoch)
print("다시 시작할 epoch:", START_EPOCH)
print("목표 epoch:", MAX_EPOCHS)
print("현재 최고 Dice:", best_metric)
print("최고 성능 epoch:", best_metric_epoch)
print("복구된 기록:", len(training_history))

마지막 정상 epoch: 20
다시 시작할 epoch: 21
목표 epoch: 50
현재 최고 Dice: 0.2556348145008087
최고 성능 epoch: 20
복구된 기록: 20


In [8]:
# 현재 50epoch 까지 학습 못함(0901진행)
# 지금 best가 20epoch로 더 이상 학습은 의미가 없어보임
%pip install -q monai nibabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.8 MB/s eta 0:00:00


In [9]:
import torch

from pathlib import Path


DATA_ROOT = Path(
    "/content/drive/MyDrive/COCA_NIFTI"
)

MODEL_DIR = (
    DATA_ROOT / "segresnet_results"
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("모델 폴더:", MODEL_DIR)
print("사용 장치:", device)
print(
    "최적 모델 존재:",
    (
        MODEL_DIR
        / "segresnet_best.pth"
    ).exists(),
)

모델 폴더: /content/drive/MyDrive/COCA_NIFTI/segresnet_results
사용 장치: cuda
최적 모델 존재: True


In [10]:
from monai.networks.nets import SegResNet


model = SegResNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,
    init_filters=16,
    blocks_down=(1, 2, 2, 4),
    blocks_up=(1, 1, 1),
    dropout_prob=0.2,
    norm="INSTANCE",
).to(device)


model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.eval()


print(
    "불러온 최고 모델 epoch:",
    best_checkpoint["epoch"],
)

print(
    "저장된 Validation Dice:",
    best_checkpoint["val_dice"],
)

print(
    "모델 장치:",
    next(model.parameters()).device,
)

불러온 최고 모델 epoch: 20
저장된 Validation Dice: 0.2556348145008087
모델 장치: cuda:0


In [11]:
import torch
best_checkpoint = torch.load(
    MODEL_DIR / "segresnet_best.pth",
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

print(
    "불러온 최고 모델 epoch:",
    best_checkpoint["epoch"],
)

print(
    "저장된 Validation Dice:",
    best_checkpoint["val_dice"],
)

불러온 최고 모델 epoch: 20
저장된 Validation Dice: 0.2556348145008087


In [12]:
import pandas as pd
import torch

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    EnsureTyped,
)

from monai.data import (
    Dataset,
    DataLoader,
)


SPACING = (0.5, 0.5, 3.0)
PATCH_SIZE = (96, 96, 32)


val_dataframe = pd.read_csv(
    DATA_ROOT / "val.csv",
    dtype={"patient_id": str},
)


val_files = []

for patient_id in val_dataframe["patient_id"]:
    val_files.append(
        {
            "image": str(
                DATA_ROOT
                / "images"
                / f"coca_{patient_id}.nii.gz"
            ),
            "label": str(
                DATA_ROOT
                / "labels"
                / f"coca_{patient_id}.nii.gz"
            ),
            "patient_id": patient_id,
        }
    )


val_transforms = Compose(
    [
        LoadImaged(
            keys=["image", "label"],
        ),

        EnsureChannelFirstd(
            keys=["image", "label"],
        ),

        Orientationd(
            keys=["image", "label"],
            axcodes="RAS",
        ),

        Spacingd(
            keys=["image", "label"],
            pixdim=SPACING,
            mode=("bilinear", "nearest"),
        ),

        ScaleIntensityRanged(
            keys=["image"],
            a_min=-200,
            a_max=1000,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),

        EnsureTyped(
            keys=["image", "label"],
            dtype=(
                torch.float32,
                torch.int64,
            ),
        ),
    ]
)


val_dataset = Dataset(
    data=val_files,
    transform=val_transforms,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)


print("검증 환자:", len(val_dataset))
print("검증 batch:", len(val_loader))
print("PATCH_SIZE:", PATCH_SIZE)

검증 환자: 43
검증 batch: 43
PATCH_SIZE: (96, 96, 32)


/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [13]:
import numpy as np
import torch

from tqdm.auto import tqdm

from monai.inferers import (
    sliding_window_inference,
)


# 정규화된 영상에서 130 HU에 해당하는 값
HU_130_NORMALIZED = (
    (130 - (-200))
    / (1000 - (-200))
)

print(
    "정규화된 130 HU 기준:",
    HU_130_NORMALIZED,
)


raw_results = []
hu130_results = []


model.eval()

torch.cuda.empty_cache()


with torch.no_grad():
    for val_data in tqdm(
        val_loader,
        desc="검증 오류 분석",
        unit="명",
    ):
        images = val_data["image"].to(
            device,
            non_blocking=True,
        )

        labels = (
            val_data["label"].to(
                device,
                non_blocking=True,
            ) > 0
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            outputs = (
                sliding_window_inference(
                    inputs=images,
                    roi_size=PATCH_SIZE,
                    sw_batch_size=8,
                    predictor=model,
                    overlap=0.25,
                    mode="gaussian",
                )
            )


        # 기본 SegResNet 예측
        predictions = (
            torch.argmax(
                outputs,
                dim=1,
                keepdim=True,
            ) > 0
        )


        # SegResNet 예측 중 130 HU 이상만 유지
        hu130_predictions = (
            predictions
            & (
                images
                >= HU_130_NORMALIZED
            )
        )


        for current_prediction, result_list in [
            (
                predictions,
                raw_results,
            ),
            (
                hu130_predictions,
                hu130_results,
            ),
        ]:
            true_positive = int(
                (
                    current_prediction
                    & labels
                ).sum()
            )

            false_positive = int(
                (
                    current_prediction
                    & ~labels
                ).sum()
            )

            false_negative = int(
                (
                    ~current_prediction
                    & labels
                ).sum()
            )


            dice = (
                2 * true_positive
                / max(
                    (
                        2 * true_positive
                        + false_positive
                        + false_negative
                    ),
                    1,
                )
            )

            precision = (
                true_positive
                / max(
                    (
                        true_positive
                        + false_positive
                    ),
                    1,
                )
            )

            recall = (
                true_positive
                / max(
                    (
                        true_positive
                        + false_negative
                    ),
                    1,
                )
            )


            result_list.append(
                {
                    "dice": dice,
                    "precision": precision,
                    "recall": recall,
                    "pred_voxels": (
                        true_positive
                        + false_positive
                    ),
                    "label_voxels": (
                        true_positive
                        + false_negative
                    ),
                }
            )


def print_metric_summary(
    name,
    results,
):
    print(f"\n{name}")

    for metric_name in [
        "dice",
        "precision",
        "recall",
        "pred_voxels",
        "label_voxels",
    ]:
        values = [
            result[metric_name]
            for result in results
        ]

        print(
            f"{metric_name}: "
            f"{np.mean(values):.4f}"
        )


print_metric_summary(
    name="기본 SegResNet 예측",
    results=raw_results,
)

print_metric_summary(
    name="130 HU 이상으로 제한한 예측",
    results=hu130_results,
)

정규화된 130 HU 기준: 0.275


검증 오류 분석:   0%|          | 0/43 [00:00<?, ?명/s]


기본 SegResNet 예측
dice: 0.2556
precision: 0.1829
recall: 0.7559
pred_voxels: 1413.2791
label_voxels: 401.9302

130 HU 이상으로 제한한 예측
dice: 0.4741
precision: 0.4170
recall: 0.7441
pred_voxels: 599.5581
label_voxels: 401.9302


실제보다 약 3.5배 많은 영역을 석회화로 예측했기 때문에 Precision과 Dice 가 낮아진 것으로 확인됨

130 HU 미만 예측은 제거하자
-> Dice, Precision 상승, Recall 값 거의 유지

실제 석회화를 거의 그대로 찾으면서 잘못 예측한 영역을 많이 제거한 것



In [21]:
%pip install -q monai SimpleITK scikit-image pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.9 MB/s eta 0:00:00


In [22]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch


# 기존 원본 HU NIfTI
SOURCE_ROOT = Path(
    "/content/drive/MyDrive/COCA_NIFTI"
)

IMAGE_DIR = (
    SOURCE_ROOT / "images"
)


# 새 공통 데이터 폴더
COMMON_ROOT = Path(
    "/content/drive/MyDrive/"
    "COCA_COMMON_MULTICLASS"
)

MULTICLASS_LABEL_DIR = (
    COMMON_ROOT / "labels_multiclass"
)

WINDOW_MANIFEST_DIR = (
    COMMON_ROOT / "manifests"
)

RESULT_DIR = (
    COMMON_ROOT / "segresnet_results"
)


for directory in [
    MULTICLASS_LABEL_DIR,
    WINDOW_MANIFEST_DIR,
    RESULT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


CROP_Y = slice(36, 436)
CROP_X = slice(62, 462)

WINDOW_DEPTH = 40
WINDOW_STRIDE = 20

CT_PADDING_VALUE = -1000.0
GT_PADDING_VALUE = 0

HU_MIN = -1000.0
HU_MAX = 2000.0

NUM_CLASSES = 5


print(
    "기존 CT:",
    len(list(IMAGE_DIR.glob("*.nii.gz"))),
)

print(
    "기존 이진 마스크:",
    len(
        list(
            (
                SOURCE_ROOT / "labels"
            ).glob("*.nii.gz")
        )
    ),
)

print(
    "새 다중 클래스 마스크:",
    len(
        list(
            MULTICLASS_LABEL_DIR.glob(
                "*.nii.gz"
            )
        )
    ),
)

기존 CT: 434
기존 이진 마스크: 434
새 다중 클래스 마스크: 0


In [25]:
import pandas as pd
from pathlib import Path


SPLIT_PATH = (
    SOURCE_ROOT / "dataset_split.csv"
)

if not SPLIT_PATH.exists():
    raise FileNotFoundError(
        f"분할 파일이 없습니다: {SPLIT_PATH}"
    )


split_dataframe = pd.read_csv(
    SPLIT_PATH,
    dtype={"patient_id": str},
)

required_columns = {
    "patient_id",
    "split",
}

missing_columns = (
    required_columns
    - set(split_dataframe.columns)
)

if missing_columns:
    raise ValueError(
        f"분할 파일에 필요한 컬럼이 없습니다: "
        f"{missing_columns}"
    )


usable_patient_ids = (
    split_dataframe["patient_id"]
    .astype(str)
    .tolist()
)


print("전체 환자:", len(usable_patient_ids))

print(
    split_dataframe["split"]
    .value_counts()
    .reindex(
        ["train", "val", "test"]
    )
)

print(
    "중복 환자:",
    split_dataframe["patient_id"]
    .duplicated()
    .sum(),
)

전체 환자: 434
split
train    347
val       43
test      44
Name: count, dtype: int64
중복 환자: 0


In [26]:
from pathlib import Path


XML_ROOT = Path(
    "/content/drive/MyDrive/"
    "COCA_RAW/Gated_release_final/calcium_xml"
)


multiclass_paths = list(
    MULTICLASS_LABEL_DIR.glob(
        "coca_*.nii.gz"
    )
)

xml_paths = list(
    XML_ROOT.glob("*.xml")
)


multiclass_ids = {
    path.name
    .removeprefix("coca_")
    .removesuffix(".nii.gz")
    for path in multiclass_paths
}

xml_ids = {
    path.stem
    for path in xml_paths
}

usable_id_set = set(
    usable_patient_ids
)


print(
    "팀 다중 클래스 마스크:",
    len(multiclass_paths),
)

print(
    "COCA XML:",
    len(xml_paths),
)

print(
    "사용 환자와 일치하는 다중 클래스 마스크:",
    len(
        usable_id_set
        & multiclass_ids
    ),
)

print(
    "사용 환자와 일치하는 XML:",
    len(
        usable_id_set
        & xml_ids
    ),
)


if usable_id_set.issubset(
    multiclass_ids
):
    ANNOTATION_SOURCE = (
        "multiclass_nifti"
    )

    print(
        "\n사용 방식: "
        "팀 다중 클래스 NIfTI"
    )

elif usable_id_set.issubset(
    xml_ids
):
    ANNOTATION_SOURCE = "xml"

    print(
        "\n사용 방식: "
        "COCA XML에서 다중 클래스 생성"
    )

else:
    ANNOTATION_SOURCE = None

    missing_multiclass = sorted(
        usable_id_set
        - multiclass_ids,
        key=int,
    )

    missing_xml = sorted(
        usable_id_set
        - xml_ids,
        key=int,
    )

    print(
        "\n다중 클래스 원본이 부족합니다."
    )

    print(
        "다중 클래스 마스크 누락:",
        len(missing_multiclass),
    )

    print(
        "XML 누락:",
        len(missing_xml),
    )

    print(
        "XML 누락 예시:",
        missing_xml[:10],
    )

팀 다중 클래스 마스크: 0
COCA XML: 0
사용 환자와 일치하는 다중 클래스 마스크: 0
사용 환자와 일치하는 XML: 0

다중 클래스 원본이 부족합니다.
다중 클래스 마스크 누락: 434
XML 누락: 434
XML 누락 예시: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


In [27]:
# 기존 이진 마스크 사용
LABEL_DIR = (
    SOURCE_ROOT / "labels"
)

# 0: Background
# 1: Coronary calcium
NUM_CLASSES = 2


print(
    "CT:",
    len(list(IMAGE_DIR.glob("*.nii.gz"))),
)

print(
    "이진 마스크:",
    len(list(LABEL_DIR.glob("*.nii.gz"))),
)

assert (
    len(list(IMAGE_DIR.glob("*.nii.gz")))
    == 434
)

assert (
    len(list(LABEL_DIR.glob("*.nii.gz")))
    == 434
)

print(
    "\n이진 SegResNet 데이터 준비 완료"
)

CT: 434
이진 마스크: 434

이진 SegResNet 데이터 준비 완료


In [28]:
import SimpleITK as sitk
import numpy as np


TEST_PATIENT_ID = "0"

ct_path = (
    IMAGE_DIR
    / f"coca_{TEST_PATIENT_ID}.nii.gz"
)

label_path = (
    LABEL_DIR
    / f"coca_{TEST_PATIENT_ID}.nii.gz"
)


ct_image = sitk.ReadImage(
    str(ct_path)
)

label_image = sitk.ReadImage(
    str(label_path)
)


# SimpleITK 배열 순서: [D, Y, X]
ct_volume = sitk.GetArrayFromImage(
    ct_image
).astype(np.float32)

label_volume = sitk.GetArrayFromImage(
    label_image
).astype(np.uint8)


print("원본 CT 크기:", ct_volume.shape)
print("원본 마스크 크기:", label_volume.shape)

print("CT spacing:", ct_image.GetSpacing())
print(
    "마스크 spacing:",
    label_image.GetSpacing(),
)

print("CT origin:", ct_image.GetOrigin())
print(
    "마스크 origin:",
    label_image.GetOrigin(),
)

print(
    "마스크 고유값:",
    np.unique(label_volume),
)

print(
    "원본 마스크 voxel:",
    int(label_volume.sum()),
)


# CT와 마스크 검증
assert (
    ct_volume.shape
    == label_volume.shape
), "CT와 마스크 크기가 다릅니다."

assert np.allclose(
    ct_image.GetSpacing(),
    label_image.GetSpacing(),
), "CT와 마스크 spacing이 다릅니다."

assert np.allclose(
    ct_image.GetOrigin(),
    label_image.GetOrigin(),
), "CT와 마스크 origin이 다릅니다."

assert np.allclose(
    ct_image.GetDirection(),
    label_image.GetDirection(),
), "CT와 마스크 direction이 다릅니다."

assert set(
    np.unique(label_volume).tolist()
).issubset({0, 1}), (
    "이진 마스크가 아닙니다."
)


# 팀 전처리와 동일한 고정 크롭
cropped_ct = ct_volume[
    :,
    CROP_Y,
    CROP_X,
]

cropped_label = label_volume[
    :,
    CROP_Y,
    CROP_X,
]


print("\n고정 크롭 결과")
print("크롭 CT 크기:", cropped_ct.shape)
print(
    "크롭 마스크 크기:",
    cropped_label.shape,
)

print(
    "크롭 후 마스크 voxel:",
    int(cropped_label.sum()),
)

print(
    "크롭으로 제외된 마스크 voxel:",
    int(
        label_volume.sum()
        - cropped_label.sum()
    ),
)


assert cropped_ct.shape[1:] == (
    400,
    400,
)

assert cropped_label.shape[1:] == (
    400,
    400,
)

print("\nCT·마스크 공간정보 및 크롭 시험 통과")

원본 CT 크기: (57, 512, 512)
원본 마스크 크기: (57, 512, 512)
CT spacing: (0.474609375, 0.474609375, 3.0)
마스크 spacing: (0.474609375, 0.474609375, 3.0)
CT origin: (-90.2626953125, -317.2626953125, -285.25)
마스크 origin: (-90.2626953125, -317.2626953125, -285.25)
마스크 고유값: [0 1]
원본 마스크 voxel: 13

고정 크롭 결과
크롭 CT 크기: (57, 400, 400)
크롭 마스크 크기: (57, 400, 400)
크롭 후 마스크 voxel: 13
크롭으로 제외된 마스크 voxel: 0

CT·마스크 공간정보 및 크롭 시험 통과


In [29]:
from tqdm.auto import tqdm
import SimpleITK as sitk
import pandas as pd
import numpy as np


split_dataframe = pd.read_csv(
    SOURCE_ROOT / "dataset_split.csv",
    dtype={"patient_id": str},
)

usable_patient_ids = (
    split_dataframe["patient_id"]
    .astype(str)
    .tolist()
)


crop_audit_results = []
crop_audit_errors = []


for patient_id in tqdm(
    usable_patient_ids,
    desc="전체 크롭 검사",
):
    try:
        label_path = (
            LABEL_DIR
            / f"coca_{patient_id}.nii.gz"
        )

        if not label_path.exists():
            raise FileNotFoundError(
                f"마스크 파일 없음: {label_path}"
            )

        label_image = sitk.ReadImage(
            str(label_path)
        )

        label_volume = (
            sitk.GetArrayFromImage(
                label_image
            )
            .astype(np.uint8)
        )

        unique_values = set(
            np.unique(label_volume).tolist()
        )

        if not unique_values.issubset(
            {0, 1}
        ):
            raise ValueError(
                f"이진 마스크가 아님: "
                f"{unique_values}"
            )

        original_voxels = int(
            np.count_nonzero(label_volume)
        )

        cropped_label = label_volume[
            :,
            CROP_Y,
            CROP_X,
        ]

        cropped_voxels = int(
            np.count_nonzero(cropped_label)
        )

        excluded_voxels = (
            original_voxels
            - cropped_voxels
        )

        crop_audit_results.append(
            {
                "patient_id": patient_id,
                "depth": label_volume.shape[0],
                "original_voxels": (
                    original_voxels
                ),
                "cropped_voxels": (
                    cropped_voxels
                ),
                "excluded_voxels": (
                    excluded_voxels
                ),
            }
        )

    except Exception as error:
        crop_audit_errors.append(
            {
                "patient_id": patient_id,
                "error": repr(error),
            }
        )


crop_audit_dataframe = pd.DataFrame(
    crop_audit_results
)

crop_audit_path = (
    WINDOW_MANIFEST_DIR
    / "crop_audit.csv"
)

crop_audit_dataframe.to_csv(
    crop_audit_path,
    index=False,
)


excluded_cases = (
    crop_audit_dataframe[
        crop_audit_dataframe[
            "excluded_voxels"
        ] > 0
    ]
)


print("\n전체 크롭 검사 완료")
print(
    "검사 성공:",
    len(crop_audit_dataframe),
)
print(
    "처리 오류:",
    len(crop_audit_errors),
)
print(
    "병변이 잘린 환자:",
    len(excluded_cases),
)
print(
    "잘린 전체 voxel:",
    int(
        crop_audit_dataframe[
            "excluded_voxels"
        ].sum()
    ),
)
print(
    "크롭 후 빈 마스크:",
    int(
        (
            crop_audit_dataframe[
                "cropped_voxels"
            ] == 0
        ).sum()
    ),
)


if len(excluded_cases) > 0:
    print("\n병변이 잘린 환자")
    print(
        excluded_cases[
            [
                "patient_id",
                "original_voxels",
                "cropped_voxels",
                "excluded_voxels",
            ]
        ].to_string(index=False)
    )


if crop_audit_errors:
    print("\n처리 오류")
    for error in crop_audit_errors:
        print(error)


print(
    "\n검사 결과 저장:",
    crop_audit_path,
)

전체 크롭 검사:   0%|          | 0/434 [00:00<?, ?it/s]


전체 크롭 검사 완료
검사 성공: 434
처리 오류: 0
병변이 잘린 환자: 0
잘린 전체 voxel: 0
크롭 후 빈 마스크: 0

검사 결과 저장: /content/drive/MyDrive/COCA_COMMON_MULTICLASS/manifests/crop_audit.csv


In [30]:
import pandas as pd


if crop_audit_errors:
    raise RuntimeError(
        "크롭 검사 오류가 있으므로 "
        "윈도우를 생성할 수 없습니다."
    )

if (
    crop_audit_dataframe[
        "excluded_voxels"
    ].sum() > 0
):
    raise RuntimeError(
        "크롭으로 제외된 병변이 있습니다. "
        "결과를 먼저 확인해야 합니다."
    )


depth_by_patient = dict(
    zip(
        crop_audit_dataframe[
            "patient_id"
        ].astype(str),
        crop_audit_dataframe[
            "depth"
        ].astype(int),
    )
)

split_by_patient = dict(
    zip(
        split_dataframe[
            "patient_id"
        ].astype(str),
        split_dataframe[
            "split"
        ].astype(str),
    )
)


window_records = []


for patient_id in usable_patient_ids:
    depth = depth_by_patient[
        patient_id
    ]

    split_name = split_by_patient[
        patient_id
    ]


    # 깊이가 40 이하이면 윈도우 1개
    if depth <= WINDOW_DEPTH:
        window_starts = [0]

    else:
        last_start = (
            depth - WINDOW_DEPTH
        )

        window_starts = list(
            range(
                0,
                last_start + 1,
                WINDOW_STRIDE,
            )
        )

        # 마지막 slice까지 빠짐없이 포함
        if window_starts[-1] != last_start:
            window_starts.append(
                last_start
            )


    for window_index, z_start in enumerate(
        window_starts
    ):
        z_end = min(
            z_start + WINDOW_DEPTH,
            depth,
        )

        padding_after = max(
            0,
            WINDOW_DEPTH - (z_end - z_start),
        )

        window_records.append(
            {
                "patient_id": patient_id,
                "split": split_name,
                "window_index": window_index,
                "depth": depth,
                "z_start": z_start,
                "z_end": z_end,
                "padding_after": (
                    padding_after
                ),
            }
        )


window_manifest = pd.DataFrame(
    window_records
)


# 전체 목록 저장
window_manifest.to_csv(
    WINDOW_MANIFEST_DIR
    / "all_windows.csv",
    index=False,
)


# 학습·검증·테스트별 저장
for split_name in [
    "train",
    "val",
    "test",
]:
    split_windows = (
        window_manifest[
            window_manifest["split"]
            == split_name
        ]
        .reset_index(drop=True)
    )

    split_windows.to_csv(
        WINDOW_MANIFEST_DIR
        / f"{split_name}_windows.csv",
        index=False,
    )

    print(
        f"{split_name}: "
        f"{split_windows['patient_id'].nunique()}명"
        f" | {len(split_windows)}개 window"
    )


print(
    "\n전체 환자:",
    window_manifest[
        "patient_id"
    ].nunique(),
)

print(
    "전체 window:",
    len(window_manifest),
)

print(
    "padding이 필요한 window:",
    int(
        (
            window_manifest[
                "padding_after"
            ] > 0
        ).sum()
    ),
)

print(
    "최대 padding:",
    int(
        window_manifest[
            "padding_after"
        ].max()
    ),
)

print(
    "\n저장 위치:",
    WINDOW_MANIFEST_DIR,
)

train: 347명 | 688개 window
val: 43명 | 84개 window
test: 44명 | 93개 window

전체 환자: 434
전체 window: 865
padding이 필요한 window: 19
최대 padding: 6

저장 위치: /content/drive/MyDrive/COCA_COMMON_MULTICLASS/manifests


In [31]:
from torch.utils.data import Dataset
import SimpleITK as sitk
import numpy as np
import torch
import pandas as pd


class COCAWindowDataset(Dataset):
    def __init__(
        self,
        manifest,
        image_dir,
        label_dir,
    ):
        self.manifest = (
            manifest
            .reset_index(drop=True)
            .copy()
        )

        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)


    def __len__(self):
        return len(self.manifest)


    def __getitem__(self, index):
        row = self.manifest.iloc[index]

        patient_id = str(
            row["patient_id"]
        )

        z_start = int(row["z_start"])
        z_end = int(row["z_end"])


        ct_path = (
            self.image_dir
            / f"coca_{patient_id}.nii.gz"
        )

        label_path = (
            self.label_dir
            / f"coca_{patient_id}.nii.gz"
        )


        # 배열 순서: [D, Y, X]
        ct_volume = (
            sitk.GetArrayFromImage(
                sitk.ReadImage(
                    str(ct_path)
                )
            )
            .astype(np.float32)
        )

        label_volume = (
            sitk.GetArrayFromImage(
                sitk.ReadImage(
                    str(label_path)
                )
            )
            .astype(np.uint8)
        )


        # 고정 XY 크롭
        ct_volume = ct_volume[
            :,
            CROP_Y,
            CROP_X,
        ]

        label_volume = label_volume[
            :,
            CROP_Y,
            CROP_X,
        ]


        # Z축 윈도우
        ct_window = ct_volume[
            z_start:z_end
        ]

        label_window = label_volume[
            z_start:z_end
        ]


        # 깊이가 40보다 짧으면 뒤쪽 padding
        padding_after = (
            WINDOW_DEPTH
            - ct_window.shape[0]
        )

        if padding_after > 0:
            ct_window = np.pad(
                ct_window,
                pad_width=(
                    (0, padding_after),
                    (0, 0),
                    (0, 0),
                ),
                mode="constant",
                constant_values=(
                    CT_PADDING_VALUE
                ),
            )

            label_window = np.pad(
                label_window,
                pad_width=(
                    (0, padding_after),
                    (0, 0),
                    (0, 0),
                ),
                mode="constant",
                constant_values=(
                    GT_PADDING_VALUE
                ),
            )


        # HU clipping
        ct_window = np.clip(
            ct_window,
            HU_MIN,
            HU_MAX,
        )


        # [-1000, 2000] -> [-1, 2]
        ct_window = (
            ct_window / 1000.0
        ).astype(np.float32)


        # 연속 메모리 배열로 변환
        ct_window = np.ascontiguousarray(
            ct_window
        )

        label_window = (
            np.ascontiguousarray(
                label_window
            )
        )


        # CT: [1, D, H, W]
        image_tensor = torch.from_numpy(
            ct_window[None, ...]
        ).float()

        # GT: [D, H, W]
        label_tensor = torch.from_numpy(
            label_window.astype(
                np.int64
            )
        ).long()


        return {
            "image": image_tensor,
            "label": label_tensor,
            "patient_id": patient_id,
            "z_start": z_start,
            "z_end": z_end,
        }

In [32]:
train_manifest = pd.read_csv(
    WINDOW_MANIFEST_DIR
    / "train_windows.csv",
    dtype={"patient_id": str},
)

train_dataset = COCAWindowDataset(
    manifest=train_manifest,
    image_dir=IMAGE_DIR,
    label_dir=LABEL_DIR,
)


sample = train_dataset[0]


print(
    "학습 window:",
    len(train_dataset),
)

print(
    "환자:",
    sample["patient_id"],
)

print(
    "Z 범위:",
    sample["z_start"],
    "~",
    sample["z_end"],
)

print(
    "CT 크기:",
    sample["image"].shape,
)

print(
    "마스크 크기:",
    sample["label"].shape,
)

print(
    "CT 값 범위:",
    float(sample["image"].min()),
    "~",
    float(sample["image"].max()),
)

print(
    "마스크 고유값:",
    torch.unique(
        sample["label"]
    ).tolist(),
)

print(
    "마스크 voxel:",
    int(
        torch.count_nonzero(
            sample["label"]
        )
    ),
)


assert sample["image"].shape == (
    1,
    40,
    400,
    400,
)

assert sample["label"].shape == (
    40,
    400,
    400,
)

assert (
    float(sample["image"].min())
    >= -1.0
)

assert (
    float(sample["image"].max())
    <= 2.0
)

print(
    "\nDataset 생성 시험 통과"
)

학습 window: 688
환자: 234
Z 범위: 0 ~ 40
CT 크기: torch.Size([1, 40, 400, 400])
마스크 크기: torch.Size([40, 400, 400])
CT 값 범위: -1.0 ~ 1.2009999752044678
마스크 고유값: [0, 1]
마스크 voxel: 145

Dataset 생성 시험 통과


In [33]:
from torch.utils.data import DataLoader
import torch
import pandas as pd


val_manifest = pd.read_csv(
    WINDOW_MANIFEST_DIR
    / "val_windows.csv",
    dtype={"patient_id": str},
)

test_manifest = pd.read_csv(
    WINDOW_MANIFEST_DIR
    / "test_windows.csv",
    dtype={"patient_id": str},
)


val_dataset = COCAWindowDataset(
    manifest=val_manifest,
    image_dir=IMAGE_DIR,
    label_dir=LABEL_DIR,
)

test_dataset = COCAWindowDataset(
    manifest=test_manifest,
    image_dir=IMAGE_DIR,
    label_dir=LABEL_DIR,
)


BATCH_SIZE = 1

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": 0,
    "pin_memory": torch.cuda.is_available(),
}


train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    drop_last=False,
    **loader_options,
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    drop_last=False,
    **loader_options,
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    drop_last=False,
    **loader_options,
)


print(
    "학습:",
    len(train_dataset),
    "window /",
    len(train_loader),
    "batch",
)

print(
    "검증:",
    len(val_dataset),
    "window /",
    len(val_loader),
    "batch",
)

print(
    "테스트:",
    len(test_dataset),
    "window /",
    len(test_loader),
    "batch",
)


# 실제 batch 1개 확인
test_batch = next(
    iter(train_loader)
)


print("\nBatch 확인")

print(
    "CT:",
    test_batch["image"].shape,
)

print(
    "마스크:",
    test_batch["label"].shape,
)

print(
    "CT dtype:",
    test_batch["image"].dtype,
)

print(
    "마스크 dtype:",
    test_batch["label"].dtype,
)

print(
    "CT 범위:",
    float(test_batch["image"].min()),
    "~",
    float(test_batch["image"].max()),
)

print(
    "마스크 고유값:",
    torch.unique(
        test_batch["label"]
    ).tolist(),
)


assert test_batch["image"].shape == (
    1,
    1,
    40,
    400,
    400,
)

assert test_batch["label"].shape == (
    1,
    40,
    400,
    400,
)

print(
    "\nDataLoader 생성 시험 통과"
)

학습: 688 window / 688 batch
검증: 84 window / 84 batch
테스트: 93 window / 93 batch

Batch 확인
CT: torch.Size([1, 1, 40, 400, 400])
마스크: torch.Size([1, 40, 400, 400])
CT dtype: torch.float32
마스크 dtype: torch.int64
CT 범위: -1.0 ~ 1.2860000133514404
마스크 고유값: [0]

DataLoader 생성 시험 통과


In [34]:
from tqdm.auto import tqdm
import SimpleITK as sitk
import numpy as np
import pandas as pd


all_window_path = (
    WINDOW_MANIFEST_DIR
    / "all_windows.csv"
)

window_manifest = pd.read_csv(
    all_window_path,
    dtype={"patient_id": str},
)

window_manifest[
    "label_voxels"
] = 0


patient_groups = (
    window_manifest
    .groupby("patient_id")
    .groups
)


for patient_id, row_indices in tqdm(
    patient_groups.items(),
    total=len(patient_groups),
    desc="윈도우 마스크 검사",
):
    label_path = (
        LABEL_DIR
        / f"coca_{patient_id}.nii.gz"
    )

    label_volume = (
        sitk.GetArrayFromImage(
            sitk.ReadImage(
                str(label_path)
            )
        )
        .astype(np.uint8)
    )

    label_volume = label_volume[
        :,
        CROP_Y,
        CROP_X,
    ]


    for row_index in row_indices:
        z_start = int(
            window_manifest.at[
                row_index,
                "z_start",
            ]
        )

        z_end = int(
            window_manifest.at[
                row_index,
                "z_end",
            ]
        )

        voxel_count = int(
            np.count_nonzero(
                label_volume[
                    z_start:z_end
                ]
            )
        )

        window_manifest.at[
            row_index,
            "label_voxels",
        ] = voxel_count


window_manifest[
    "is_positive"
] = (
    window_manifest[
        "label_voxels"
    ] > 0
)


# 전체 및 split별 manifest 다시 저장
window_manifest.to_csv(
    all_window_path,
    index=False,
)


for split_name in [
    "train",
    "val",
    "test",
]:
    split_windows = (
        window_manifest[
            window_manifest["split"]
            == split_name
        ]
        .reset_index(drop=True)
    )

    split_windows.to_csv(
        WINDOW_MANIFEST_DIR
        / f"{split_name}_windows.csv",
        index=False,
    )

    positive_count = int(
        split_windows[
            "is_positive"
        ].sum()
    )

    negative_count = (
        len(split_windows)
        - positive_count
    )

    print(f"\n{split_name}")
    print(
        "전체 window:",
        len(split_windows),
    )
    print(
        "석회화 포함:",
        positive_count,
    )
    print(
        "빈 window:",
        negative_count,
    )
    print(
        "양성 비율:",
        f"{positive_count / len(split_windows):.1%}",
    )

    positive_voxels = (
        split_windows.loc[
            split_windows["is_positive"],
            "label_voxels",
        ]
    )

    if len(positive_voxels) > 0:
        print(
            "양성 voxel 중앙값:",
            float(
                positive_voxels.median()
            ),
        )


print(
    "\n윈도우 마스크 분포 검사 완료"
)

윈도우 마스크 검사:   0%|          | 0/434 [00:00<?, ?it/s]


train
전체 window: 688
석회화 포함: 653
빈 window: 35
양성 비율: 94.9%
양성 voxel 중앙값: 245.0

val
전체 window: 84
석회화 포함: 80
빈 window: 4
양성 비율: 95.2%
양성 voxel 중앙값: 319.0

test
전체 window: 93
석회화 포함: 86
빈 window: 7
양성 비율: 92.5%
양성 voxel 중앙값: 205.0

윈도우 마스크 분포 검사 완료


In [35]:
from monai.networks.nets import SegResNet
from monai.losses import DiceCELoss

import pandas as pd
import torch


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("사용 장치:", device)

if device.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )


# T4와 400×400×40 입력을 고려해
# init_filters=8로 시작
model = SegResNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=NUM_CLASSES,
    init_filters=8,
    dropout_prob=0.2,
    blocks_down=(1, 2, 2, 4),
    blocks_up=(1, 1, 1),
    norm=(
        "GROUP",
        {"num_groups": 8},
    ),
).to(device)


# 배경을 제외하고 석회화 Dice 계산
loss_function = DiceCELoss(
    include_background=False,
    to_onehot_y=True,
    softmax=True,
    lambda_dice=1.0,
    lambda_ce=1.0,
)


LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-5

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda"),
)


parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    "모델 파라미터:",
    f"{parameter_count:,}",
)

사용 장치: cuda
GPU: Tesla T4
모델 파라미터: 1,176,186


In [36]:
# 갱신된 manifest 다시 불러오기
train_manifest = pd.read_csv(
    WINDOW_MANIFEST_DIR
    / "train_windows.csv",
    dtype={"patient_id": str},
)

train_dataset = COCAWindowDataset(
    manifest=train_manifest,
    image_dir=IMAGE_DIR,
    label_dir=LABEL_DIR,
)


positive_indices = (
    train_manifest.index[
        train_manifest["is_positive"]
    ].tolist()
)

positive_sample = train_dataset[
    positive_indices[0]
]


# Batch 차원 추가
memory_test_images = (
    positive_sample["image"]
    .unsqueeze(0)
    .to(device)
)

memory_test_labels = (
    positive_sample["label"]
    .unsqueeze(0)
    .to(device)
)


if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


model.train()

optimizer.zero_grad(
    set_to_none=True,
)


with torch.autocast(
    device_type=device.type,
    dtype=torch.float16,
    enabled=(device.type == "cuda"),
):
    memory_test_outputs = model(
        memory_test_images
    )

    # DiceCELoss는 GT channel 차원이 필요함
    memory_test_loss = loss_function(
        memory_test_outputs,
        memory_test_labels.unsqueeze(1),
    )


# 역전파 메모리까지 확인
scaler.scale(
    memory_test_loss
).backward()


print("\n메모리 시험 결과")

print(
    "입력:",
    memory_test_images.shape,
)

print(
    "출력:",
    memory_test_outputs.shape,
)

print(
    "마스크 voxel:",
    int(
        torch.count_nonzero(
            memory_test_labels
        )
    ),
)

print(
    "시험 Loss:",
    float(
        memory_test_loss.detach()
    ),
)


if device.type == "cuda":
    print(
        "현재 GPU 메모리:",
        round(
            torch.cuda.memory_allocated()
            / 1024**3,
            2,
        ),
        "GB",
    )

    print(
        "최대 GPU 메모리:",
        round(
            torch.cuda.max_memory_allocated()
            / 1024**3,
            2,
        ),
        "GB",
    )


# 시험에서는 optimizer.step()을 하지 않음
optimizer.zero_grad(
    set_to_none=True,
)

del memory_test_images
del memory_test_labels
del memory_test_outputs

if device.type == "cuda":
    torch.cuda.empty_cache()


print(
    "\nSegResNet 순전파·역전파 시험 통과"
)


메모리 시험 결과
입력: torch.Size([1, 1, 40, 400, 400])
출력: torch.Size([1, 2, 40, 400, 400])
마스크 voxel: 145
시험 Loss: 1.7427425384521484
현재 GPU 메모리: 0.31 GB
최대 GPU 메모리: 4.11 GB

SegResNet 순전파·역전파 시험 통과


In [37]:
from tqdm.auto import tqdm
import SimpleITK as sitk
import numpy as np
import pandas as pd
import torch
import time


val_manifest = pd.read_csv(
    WINDOW_MANIFEST_DIR
    / "val_windows.csv",
    dtype={"patient_id": str},
)


def make_model_input(
    cropped_ct,
    z_start,
    z_end,
):
    ct_window = cropped_ct[
        z_start:z_end
    ]

    actual_depth = ct_window.shape[0]

    padding_after = (
        WINDOW_DEPTH
        - actual_depth
    )

    if padding_after > 0:
        ct_window = np.pad(
            ct_window,
            (
                (0, padding_after),
                (0, 0),
                (0, 0),
            ),
            mode="constant",
            constant_values=(
                CT_PADDING_VALUE
            ),
        )

    ct_window = np.clip(
        ct_window,
        HU_MIN,
        HU_MAX,
    )

    ct_window = (
        ct_window / 1000.0
    ).astype(np.float32)

    ct_window = np.ascontiguousarray(
        ct_window
    )

    # [1, 1, D, H, W]
    return (
        torch.from_numpy(
            ct_window[None, None, ...]
        )
        .float()
    )


def calculate_binary_metrics(
    prediction,
    target,
):
    prediction = prediction.astype(
        bool
    )

    target = target.astype(bool)

    true_positive = int(
        np.logical_and(
            prediction,
            target,
        ).sum()
    )

    prediction_voxels = int(
        prediction.sum()
    )

    target_voxels = int(
        target.sum()
    )

    dice_denominator = (
        prediction_voxels
        + target_voxels
    )

    dice = (
        2.0 * true_positive
        / dice_denominator
        if dice_denominator > 0
        else 1.0
    )

    precision = (
        true_positive
        / prediction_voxels
        if prediction_voxels > 0
        else 0.0
    )

    recall = (
        true_positive
        / target_voxels
        if target_voxels > 0
        else 1.0
    )

    return {
        "dice": dice,
        "precision": precision,
        "recall": recall,
        "prediction_voxels": (
            prediction_voxels
        ),
        "target_voxels": (
            target_voxels
        ),
    }


@torch.no_grad()
def validate_model(
    max_patients=None,
    show_progress=True,
):
    model.eval()

    validation_start = (
        time.perf_counter()
    )

    patient_ids = (
        val_manifest["patient_id"]
        .drop_duplicates()
        .tolist()
    )

    if max_patients is not None:
        patient_ids = patient_ids[
            :max_patients
        ]

    patient_results = []


    iterator = patient_ids

    if show_progress:
        iterator = tqdm(
            patient_ids,
            desc="환자 단위 검증",
        )


    for patient_id in iterator:
        ct_path = (
            IMAGE_DIR
            / f"coca_{patient_id}.nii.gz"
        )

        label_path = (
            LABEL_DIR
            / f"coca_{patient_id}.nii.gz"
        )


        ct_volume = (
            sitk.GetArrayFromImage(
                sitk.ReadImage(
                    str(ct_path)
                )
            )
            .astype(np.float32)
        )

        label_volume = (
            sitk.GetArrayFromImage(
                sitk.ReadImage(
                    str(label_path)
                )
            )
            .astype(np.uint8)
        )


        cropped_ct = ct_volume[
            :,
            CROP_Y,
            CROP_X,
        ]

        cropped_label = label_volume[
            :,
            CROP_Y,
            CROP_X,
        ]


        depth = cropped_ct.shape[0]

        probability_sum = np.zeros(
            cropped_ct.shape,
            dtype=np.float32,
        )

        slice_counts = np.zeros(
            depth,
            dtype=np.float32,
        )


        patient_windows = (
            val_manifest[
                val_manifest[
                    "patient_id"
                ] == patient_id
            ]
            .sort_values("window_index")
        )


        for _, row in (
            patient_windows.iterrows()
        ):
            z_start = int(
                row["z_start"]
            )

            z_end = int(
                row["z_end"]
            )

            actual_depth = (
                z_end - z_start
            )

            model_input = (
                make_model_input(
                    cropped_ct,
                    z_start,
                    z_end,
                )
                .to(
                    device,
                    non_blocking=True,
                )
            )


            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(
                    device.type == "cuda"
                ),
            ):
                output = model(
                    model_input
                )

                foreground_probability = (
                    torch.softmax(
                        output,
                        dim=1,
                    )[:, 1]
                )


            foreground_probability = (
                foreground_probability[
                    0,
                    :actual_depth,
                ]
                .float()
                .cpu()
                .numpy()
            )

            probability_sum[
                z_start:z_end
            ] += foreground_probability

            slice_counts[
                z_start:z_end
            ] += 1.0


        if np.any(slice_counts == 0):
            raise RuntimeError(
                f"예측되지 않은 slice가 있습니다: "
                f"환자 {patient_id}"
            )


        average_probability = (
            probability_sum
            / slice_counts[:, None, None]
        )


        # Softmax argmax와 동일한 0.5 기준
        raw_prediction = (
            average_probability >= 0.5
        )

        target = (
            cropped_label > 0
        )


        # CT 원본 HU 130 이상으로 제한
        hu130_prediction = (
            raw_prediction
            & (cropped_ct >= 130)
        )


        raw_metrics = (
            calculate_binary_metrics(
                raw_prediction,
                target,
            )
        )

        hu130_metrics = (
            calculate_binary_metrics(
                hu130_prediction,
                target,
            )
        )


        patient_results.append(
            {
                "patient_id": patient_id,
                "raw_dice": (
                    raw_metrics["dice"]
                ),
                "raw_precision": (
                    raw_metrics["precision"]
                ),
                "raw_recall": (
                    raw_metrics["recall"]
                ),
                "hu130_dice": (
                    hu130_metrics["dice"]
                ),
                "hu130_precision": (
                    hu130_metrics[
                        "precision"
                    ]
                ),
                "hu130_recall": (
                    hu130_metrics["recall"]
                ),
            }
        )


    results_dataframe = pd.DataFrame(
        patient_results
    )

    validation_minutes = (
        time.perf_counter()
        - validation_start
    ) / 60


    summary = {
        "raw_dice": float(
            results_dataframe[
                "raw_dice"
            ].mean()
        ),
        "raw_precision": float(
            results_dataframe[
                "raw_precision"
            ].mean()
        ),
        "raw_recall": float(
            results_dataframe[
                "raw_recall"
            ].mean()
        ),
        "hu130_dice": float(
            results_dataframe[
                "hu130_dice"
            ].mean()
        ),
        "hu130_precision": float(
            results_dataframe[
                "hu130_precision"
            ].mean()
        ),
        "hu130_recall": float(
            results_dataframe[
                "hu130_recall"
            ].mean()
        ),
        "minutes": validation_minutes,
    }

    return summary, results_dataframe

In [38]:
test_summary, test_results = (
    validate_model(
        max_patients=1,
        show_progress=True,
    )
)

print("\n검증 함수 시험 결과")
print(test_summary)
print(test_results)

print(
    "\n환자 단위 검증 함수 생성 완료"
)

환자 단위 검증:   0%|          | 0/1 [00:00<?, ?it/s]


검증 함수 시험 결과
{'raw_dice': 0.0, 'raw_precision': 0.0, 'raw_recall': 0.0, 'hu130_dice': 0.0, 'hu130_precision': 0.0, 'hu130_recall': 0.0, 'minutes': 0.05035332470000261}
  patient_id  raw_dice  raw_precision  raw_recall  hu130_dice  \
0         97       0.0            0.0         0.0         0.0   

   hu130_precision  hu130_recall  
0              0.0           0.0  

환자 단위 검증 함수 생성 완료


In [39]:
from tqdm.auto import tqdm
from pathlib import Path

import pandas as pd
import torch
import time


# 이진 모델 결과 폴더
RESULT_DIR = Path(
    "/content/drive/MyDrive/"
    "COCA_COMMON_BINARY/"
    "segresnet_results"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


MAX_EPOCHS = 50

scheduler = (
    torch.optim.lr_scheduler
    .CosineAnnealingLR(
        optimizer,
        T_max=MAX_EPOCHS,
        eta_min=1e-6,
    )
)


if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


epoch_start = time.perf_counter()

model.train()

epoch_loss_sum = 0.0


progress_bar = tqdm(
    train_loader,
    total=len(train_loader),
    desc="Epoch 1 학습",
)


for step, batch_data in enumerate(
    progress_bar,
    start=1,
):
    images = batch_data[
        "image"
    ].to(
        device,
        non_blocking=True,
    )

    labels = batch_data[
        "label"
    ].to(
        device,
        non_blocking=True,
    )


    optimizer.zero_grad(
        set_to_none=True,
    )


    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=(device.type == "cuda"),
    ):
        outputs = model(images)

        loss = loss_function(
            outputs,
            labels.unsqueeze(1),
        )


    scaler.scale(loss).backward()

    scaler.step(optimizer)

    scaler.update()


    loss_value = float(
        loss.detach()
    )

    epoch_loss_sum += loss_value


    progress_bar.set_postfix(
        loss=f"{loss_value:.4f}",
        average=(
            f"{epoch_loss_sum / step:.4f}"
        ),
    )


average_train_loss = (
    epoch_loss_sum
    / len(train_loader)
)

scheduler.step()


training_minutes = (
    time.perf_counter()
    - epoch_start
) / 60


print("\n1 epoch 학습 완료")
print(
    "평균 Train Loss:",
    average_train_loss,
)
print(
    "학습 시간:",
    round(training_minutes, 1),
    "분",
)


# 43명 전체 환자 단위 검증
validation_summary, validation_results = (
    validate_model(
        max_patients=None,
        show_progress=True,
    )
)


print("\n1 epoch 검증 결과")

print(
    "기본 Dice:",
    round(
        validation_summary[
            "raw_dice"
        ],
        4,
    ),
)

print(
    "기본 Precision:",
    round(
        validation_summary[
            "raw_precision"
        ],
        4,
    ),
)

print(
    "기본 Recall:",
    round(
        validation_summary[
            "raw_recall"
        ],
        4,
    ),
)

print(
    "HU 130 Dice:",
    round(
        validation_summary[
            "hu130_dice"
        ],
        4,
    ),
)

print(
    "HU 130 Precision:",
    round(
        validation_summary[
            "hu130_precision"
        ],
        4,
    ),
)

print(
    "HU 130 Recall:",
    round(
        validation_summary[
            "hu130_recall"
        ],
        4,
    ),
)

print(
    "검증 시간:",
    round(
        validation_summary["minutes"],
        1,
    ),
    "분",
)


if device.type == "cuda":
    print(
        "최대 GPU 메모리:",
        round(
            torch.cuda.max_memory_allocated()
            / 1024**3,
            2,
        ),
        "GB",
    )


# 1 epoch 체크포인트 저장
checkpoint_path = (
    RESULT_DIR
    / "segresnet_epoch1.pth"
)

torch.save(
    {
        "epoch": 1,
        "model_state_dict": (
            model.state_dict()
        ),
        "optimizer_state_dict": (
            optimizer.state_dict()
        ),
        "scheduler_state_dict": (
            scheduler.state_dict()
        ),
        "scaler_state_dict": (
            scaler.state_dict()
        ),
        "train_loss": (
            average_train_loss
        ),
        "raw_val_dice": (
            validation_summary[
                "raw_dice"
            ]
        ),
        "hu130_val_dice": (
            validation_summary[
                "hu130_dice"
            ]
        ),
        "crop_y": (36, 436),
        "crop_x": (62, 462),
        "window_depth": WINDOW_DEPTH,
        "window_stride": WINDOW_STRIDE,
        "hu_min": HU_MIN,
        "hu_max": HU_MAX,
        "num_classes": NUM_CLASSES,
    },
    checkpoint_path,
)


validation_results.to_csv(
    RESULT_DIR
    / "validation_epoch1.csv",
    index=False,
)


training_history = [
    {
        "epoch": 1,
        "train_loss": (
            average_train_loss
        ),
        "raw_val_dice": (
            validation_summary[
                "raw_dice"
            ]
        ),
        "hu130_val_dice": (
            validation_summary[
                "hu130_dice"
            ]
        ),
        "learning_rate": (
            optimizer.param_groups[0]["lr"]
        ),
    }
]


pd.DataFrame(
    training_history
).to_csv(
    RESULT_DIR
    / "training_history.csv",
    index=False,
)


print(
    "\n체크포인트:",
    checkpoint_path,
)

Epoch 1 학습:   0%|          | 0/688 [00:00<?, ?it/s]


1 epoch 학습 완료
평균 Train Loss: 1.3847033138885054
학습 시간: 12.2 분


환자 단위 검증:   0%|          | 0/43 [00:00<?, ?it/s]


1 epoch 검증 결과
기본 Dice: 0.0
기본 Precision: 0.0
기본 Recall: 0.0
HU 130 Dice: 0.0
HU 130 Precision: 0.0
HU 130 Recall: 0.0
검증 시간: 0.7 분
최대 GPU 메모리: 4.11 GB

체크포인트: /content/drive/MyDrive/COCA_COMMON_BINARY/segresnet_results/segresnet_epoch1.pth


In [40]:
import pandas as pd
import torch
import numpy as np


train_manifest = pd.read_csv(
    WINDOW_MANIFEST_DIR
    / "train_windows.csv",
    dtype={"patient_id": str},
)


total_window_voxels = (
    len(train_manifest)
    * WINDOW_DEPTH
    * 400
    * 400
)

total_positive_voxels = int(
    train_manifest[
        "label_voxels"
    ].sum()
)

positive_ratio = (
    total_positive_voxels
    / total_window_voxels
)


print("전체 학습 voxel:", total_window_voxels)
print(
    "석회화 voxel:",
    total_positive_voxels,
)
print(
    "석회화 비율:",
    f"{positive_ratio:.6%}",
)


# 양성 검증 윈도우 선택
val_manifest = pd.read_csv(
    WINDOW_MANIFEST_DIR
    / "val_windows.csv",
    dtype={"patient_id": str},
)

val_dataset = COCAWindowDataset(
    manifest=val_manifest,
    image_dir=IMAGE_DIR,
    label_dir=LABEL_DIR,
)

positive_index = (
    val_manifest.index[
        val_manifest["is_positive"]
    ].tolist()[0]
)

sample = val_dataset[
    positive_index
]

images = (
    sample["image"]
    .unsqueeze(0)
    .to(device)
)

target = (
    sample["label"]
    .cpu()
    .numpy()
    .astype(bool)
)


model.eval()

with torch.no_grad():
    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=(device.type == "cuda"),
    ):
        outputs = model(images)

        foreground_probability = (
            torch.softmax(
                outputs,
                dim=1,
            )[0, 1]
        )

foreground_probability = (
    foreground_probability
    .float()
    .cpu()
    .numpy()
)


print("\nForeground 확률")
print(
    "최소:",
    float(
        foreground_probability.min()
    ),
)
print(
    "평균:",
    float(
        foreground_probability.mean()
    ),
)
print(
    "최대:",
    float(
        foreground_probability.max()
    ),
)

print(
    "99.9 percentile:",
    float(
        np.percentile(
            foreground_probability,
            99.9,
        )
    ),
)


print("\nThreshold별 결과")

for threshold in [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
]:
    prediction = (
        foreground_probability
        >= threshold
    )

    intersection = int(
        np.logical_and(
            prediction,
            target,
        ).sum()
    )

    denominator = (
        int(prediction.sum())
        + int(target.sum())
    )

    dice = (
        2 * intersection
        / denominator
        if denominator > 0
        else 1.0
    )

    print(
        f"threshold={threshold:.1f}"
        f" | 예측 voxel={int(prediction.sum())}"
        f" | Dice={dice:.4f}"
    )

전체 학습 voxel: 4403200000
석회화 voxel: 522087
석회화 비율: 0.011857%

Foreground 확률
최소: 0.05813159421086311
평균: 0.21107442677021027
최대: 0.41774648427963257
99.9 percentile: 0.29854246973991394

Threshold별 결과
threshold=0.1 | 예측 voxel=6399645 | Dice=0.0000
threshold=0.2 | 예측 voxel=2788482 | Dice=0.0000
threshold=0.3 | 예측 voxel=5455 | Dice=0.0000
threshold=0.4 | 예측 voxel=7 | Dice=0.0000
threshold=0.5 | 예측 voxel=0 | Dice=0.0000


In [41]:
from monai.networks.nets import SegResNet
from monai.losses import TverskyLoss

import torch
import torch.nn as nn
import numpy as np
import random


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# 불균형 비율의 제곱근을 CE 가중치로 사용
negative_voxels = (
    total_window_voxels
    - total_positive_voxels
)

imbalance_ratio = (
    negative_voxels
    / total_positive_voxels
)

FOREGROUND_CE_WEIGHT = float(
    min(
        100.0,
        np.sqrt(imbalance_ratio),
    )
)

print(
    "Background 대비 foreground 비율:",
    round(imbalance_ratio, 1),
)

print(
    "Foreground CE 가중치:",
    round(FOREGROUND_CE_WEIGHT, 2),
)


class TverskyWeightedCELoss(
    nn.Module
):
    def __init__(
        self,
        foreground_weight,
        ce_ratio=0.5,
    ):
        super().__init__()

        # False Negative를 더 강하게 반영
        self.tversky_loss = (
            TverskyLoss(
                include_background=False,
                to_onehot_y=True,
                softmax=True,
                alpha=0.3,
                beta=0.7,
                smooth_nr=1e-5,
                smooth_dr=1e-5,
            )
        )

        class_weights = torch.tensor(
            [
                1.0,
                foreground_weight,
            ],
            dtype=torch.float32,
        )

        self.ce_loss = (
            nn.CrossEntropyLoss(
                weight=class_weights,
            )
        )

        self.ce_ratio = ce_ratio


    def forward(
        self,
        outputs,
        labels,
    ):
        # labels: [B, 1, D, H, W]
        tversky_value = (
            self.tversky_loss(
                outputs,
                labels,
            )
        )

        ce_value = self.ce_loss(
            outputs,
            labels[:, 0].long(),
        )

        return (
            tversky_value
            + self.ce_ratio * ce_value
        )


# 기존 모델을 이어서 쓰지 않고 재초기화
model = SegResNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,
    init_filters=8,
    dropout_prob=0.2,
    blocks_down=(1, 2, 2, 4),
    blocks_up=(1, 1, 1),
    norm=(
        "GROUP",
        {"num_groups": 8},
    ),
).to(device)


loss_function = (
    TverskyWeightedCELoss(
        foreground_weight=(
            FOREGROUND_CE_WEIGHT
        ),
        ce_ratio=0.5,
    )
    .to(device)
)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-5,
)


MAX_EPOCHS = 50

scheduler = (
    torch.optim.lr_scheduler
    .CosineAnnealingLR(
        optimizer,
        T_max=MAX_EPOCHS,
        eta_min=1e-6,
    )
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda"),
)


print("\n새 모델 초기화 완료")
print(
    "손실함수: "
    "Tversky + Weighted CE"
)
print(
    "기존 epoch 1 가중치 사용:",
    False,
)

Background 대비 foreground 비율: 8432.8
Foreground CE 가중치: 91.83

새 모델 초기화 완료
손실함수: Tversky + Weighted CE
기존 epoch 1 가중치 사용: False


In [42]:
from tqdm.auto import tqdm

import pandas as pd
import torch
import time


if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


epoch_start = time.perf_counter()

model.train()
epoch_loss_sum = 0.0


progress_bar = tqdm(
    train_loader,
    total=len(train_loader),
    desc="Tversky Epoch 1 학습",
)


for step, batch_data in enumerate(
    progress_bar,
    start=1,
):
    images = batch_data[
        "image"
    ].to(
        device,
        non_blocking=True,
    )

    labels = batch_data[
        "label"
    ].to(
        device,
        non_blocking=True,
    )


    optimizer.zero_grad(
        set_to_none=True,
    )


    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=(device.type == "cuda"),
    ):
        outputs = model(images)

        loss = loss_function(
            outputs,
            labels.unsqueeze(1),
        )


    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()


    loss_value = float(
        loss.detach()
    )

    epoch_loss_sum += loss_value


    progress_bar.set_postfix(
        loss=f"{loss_value:.4f}",
        average=(
            f"{epoch_loss_sum / step:.4f}"
        ),
    )


average_train_loss = (
    epoch_loss_sum
    / len(train_loader)
)

scheduler.step()


training_minutes = (
    time.perf_counter()
    - epoch_start
) / 60


print("\nTversky Epoch 1 학습 완료")
print(
    "평균 Train Loss:",
    average_train_loss,
)
print(
    "학습 시간:",
    round(training_minutes, 1),
    "분",
)


# 전체 검증 환자 43명
validation_summary, validation_results = (
    validate_model(
        max_patients=None,
        show_progress=True,
    )
)


print("\nTversky Epoch 1 검증 결과")

print(
    "기본 Dice:",
    round(
        validation_summary[
            "raw_dice"
        ],
        4,
    ),
)

print(
    "기본 Precision:",
    round(
        validation_summary[
            "raw_precision"
        ],
        4,
    ),
)

print(
    "기본 Recall:",
    round(
        validation_summary[
            "raw_recall"
        ],
        4,
    ),
)

print(
    "HU 130 Dice:",
    round(
        validation_summary[
            "hu130_dice"
        ],
        4,
    ),
)

print(
    "HU 130 Precision:",
    round(
        validation_summary[
            "hu130_precision"
        ],
        4,
    ),
)

print(
    "HU 130 Recall:",
    round(
        validation_summary[
            "hu130_recall"
        ],
        4,
    ),
)

print(
    "검증 시간:",
    round(
        validation_summary["minutes"],
        1,
    ),
    "분",
)


if device.type == "cuda":
    print(
        "최대 GPU 메모리:",
        round(
            torch.cuda.max_memory_allocated()
            / 1024**3,
            2,
        ),
        "GB",
    )


checkpoint_path = (
    RESULT_DIR
    / (
        "segresnet_"
        "tversky_weighted_"
        "epoch1.pth"
    )
)


torch.save(
    {
        "epoch": 1,
        "model_state_dict": (
            model.state_dict()
        ),
        "optimizer_state_dict": (
            optimizer.state_dict()
        ),
        "scheduler_state_dict": (
            scheduler.state_dict()
        ),
        "scaler_state_dict": (
            scaler.state_dict()
        ),
        "train_loss": (
            average_train_loss
        ),
        "raw_val_dice": (
            validation_summary[
                "raw_dice"
            ]
        ),
        "hu130_val_dice": (
            validation_summary[
                "hu130_dice"
            ]
        ),
        "foreground_ce_weight": (
            FOREGROUND_CE_WEIGHT
        ),
        "tversky_alpha": 0.3,
        "tversky_beta": 0.7,
        "crop_y": (36, 436),
        "crop_x": (62, 462),
        "window_depth": WINDOW_DEPTH,
        "window_stride": WINDOW_STRIDE,
        "num_classes": 2,
    },
    checkpoint_path,
)


validation_results.to_csv(
    RESULT_DIR
    / (
        "validation_"
        "tversky_weighted_"
        "epoch1.csv"
    ),
    index=False,
)


training_history = [
    {
        "epoch": 1,
        "train_loss": (
            average_train_loss
        ),
        "raw_val_dice": (
            validation_summary[
                "raw_dice"
            ]
        ),
        "hu130_val_dice": (
            validation_summary[
                "hu130_dice"
            ]
        ),
        "learning_rate": (
            optimizer.param_groups[0]["lr"]
        ),
    }
]


pd.DataFrame(
    training_history
).to_csv(
    RESULT_DIR
    / (
        "training_history_"
        "tversky_weighted.csv"
    ),
    index=False,
)


print(
    "\n새 체크포인트:",
    checkpoint_path,
)

Tversky Epoch 1 학습:   0%|          | 0/688 [00:00<?, ?it/s]


Tversky Epoch 1 학습 완료
평균 Train Loss: 1.2772008634583896
학습 시간: 11.4 분


환자 단위 검증:   0%|          | 0/43 [00:00<?, ?it/s]


Tversky Epoch 1 검증 결과
기본 Dice: 0.0006
기본 Precision: 0.0004
기본 Recall: 0.0011
HU 130 Dice: 0.0013
HU 130 Precision: 0.0027
HU 130 Recall: 0.0011
검증 시간: 0.7 분
최대 GPU 메모리: 4.12 GB

새 체크포인트: /content/drive/MyDrive/COCA_COMMON_BINARY/segresnet_results/segresnet_tversky_weighted_epoch1.pth


In [43]:
from tqdm.auto import tqdm

import copy
import numpy as np
import torch


# 석회화 voxel이 가장 많은 학습 window 선택
sanity_index = int(
    train_manifest[
        "label_voxels"
    ].idxmax()
)

sanity_sample = train_dataset[
    sanity_index
]

sanity_images = (
    sanity_sample["image"]
    .unsqueeze(0)
    .to(device)
)

sanity_labels = (
    sanity_sample["label"]
    .unsqueeze(0)
    .to(device)
)


print(
    "시험 환자:",
    sanity_sample["patient_id"],
)

print(
    "마스크 voxel:",
    int(
        torch.count_nonzero(
            sanity_labels
        )
    ),
)


# 현재 본 모델은 그대로 보존
sanity_model = copy.deepcopy(
    model
).to(device)

sanity_optimizer = (
    torch.optim.AdamW(
        sanity_model.parameters(),
        lr=1e-3,
        weight_decay=0.0,
    )
)

sanity_scaler = (
    torch.amp.GradScaler(
        "cuda",
        enabled=(
            device.type == "cuda"
        ),
    )
)


sanity_model.train()

progress_bar = tqdm(
    range(1, 101),
    desc="단일 window 과적합 시험",
)


for step in progress_bar:
    sanity_optimizer.zero_grad(
        set_to_none=True,
    )

    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=(device.type == "cuda"),
    ):
        sanity_outputs = sanity_model(
            sanity_images
        )

        sanity_loss = loss_function(
            sanity_outputs,
            sanity_labels.unsqueeze(1),
        )

    sanity_scaler.scale(
        sanity_loss
    ).backward()

    sanity_scaler.step(
        sanity_optimizer
    )

    sanity_scaler.update()

    progress_bar.set_postfix(
        loss=(
            f"{float(sanity_loss.detach()):.4f}"
        )
    )


# 같은 데이터 예측
sanity_model.eval()

with torch.no_grad():
    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=(device.type == "cuda"),
    ):
        sanity_outputs = sanity_model(
            sanity_images
        )

        sanity_probability = (
            torch.softmax(
                sanity_outputs,
                dim=1,
            )[0, 1]
        )


sanity_prediction = (
    sanity_probability >= 0.5
)

sanity_target = (
    sanity_labels[0] > 0
)


true_positive = int(
    torch.logical_and(
        sanity_prediction,
        sanity_target,
    ).sum()
)

prediction_voxels = int(
    sanity_prediction.sum()
)

target_voxels = int(
    sanity_target.sum()
)

sanity_dice = (
    2.0 * true_positive
    / (
        prediction_voxels
        + target_voxels
    )
    if (
        prediction_voxels
        + target_voxels
    ) > 0
    else 1.0
)


# 정규화 값 0.13 = 원본 130 HU
hu130_prediction = (
    sanity_prediction
    & (
        sanity_images[
            0,
            0,
        ] >= 0.13
    )
)

hu130_true_positive = int(
    torch.logical_and(
        hu130_prediction,
        sanity_target,
    ).sum()
)

hu130_prediction_voxels = int(
    hu130_prediction.sum()
)

hu130_dice = (
    2.0 * hu130_true_positive
    / (
        hu130_prediction_voxels
        + target_voxels
    )
    if (
        hu130_prediction_voxels
        + target_voxels
    ) > 0
    else 1.0
)


print("\n단일 window 과적합 결과")
print(
    "Foreground 최대 확률:",
    float(
        sanity_probability.max()
    ),
)
print(
    "예측 voxel:",
    prediction_voxels,
)
print(
    "정답 voxel:",
    target_voxels,
)
print(
    "기본 Dice:",
    round(sanity_dice, 4),
)
print(
    "HU 130 Dice:",
    round(hu130_dice, 4),
)


# 시험 모델만 제거
del sanity_model
del sanity_optimizer
del sanity_scaler
del sanity_images
del sanity_labels
del sanity_outputs
del sanity_probability

if device.type == "cuda":
    torch.cuda.empty_cache()

시험 환자: 321
마스크 voxel: 9962


단일 window 과적합 시험:   0%|          | 0/100 [00:00<?, ?it/s]


단일 window 과적합 결과
Foreground 최대 확률: 0.9999995231628418
예측 voxel: 52138
정답 voxel: 9962
기본 Dice: 0.3194
HU 130 Dice: 0.6406


In [44]:
from tqdm.auto import tqdm
from pathlib import Path

import pandas as pd
import torch
import time


START_EPOCH = 2
FINAL_EPOCH = 20

history_path = (
    RESULT_DIR
    / (
        "training_history_"
        "tversky_weighted.csv"
    )
)

last_model_path = (
    RESULT_DIR
    / "segresnet_last.pth"
)

best_model_path = (
    RESULT_DIR
    / "segresnet_best_hu130.pth"
)


# epoch 1 기록 불러오기
if not history_path.exists():
    raise FileNotFoundError(
        f"epoch 1 기록이 없습니다: "
        f"{history_path}"
    )


history_dataframe = pd.read_csv(
    history_path
)

training_history = (
    history_dataframe
    .to_dict("records")
)


if 1 not in (
    history_dataframe["epoch"]
    .astype(int)
    .tolist()
):
    raise ValueError(
        "epoch 1 학습 기록을 찾지 못했습니다."
    )


# 현재 최고 성능 복구
best_row_index = (
    history_dataframe[
        "hu130_val_dice"
    ].astype(float).idxmax()
)

best_metric = float(
    history_dataframe.loc[
        best_row_index,
        "hu130_val_dice",
    ]
)

best_metric_epoch = int(
    history_dataframe.loc[
        best_row_index,
        "epoch",
    ]
)


print("학습 시작 epoch:", START_EPOCH)
print("최종 epoch:", FINAL_EPOCH)
print(
    "현재 최고 HU 130 Dice:",
    best_metric,
)
print(
    "현재 최고 epoch:",
    best_metric_epoch,
)


def make_checkpoint(
    epoch,
    train_loss,
    raw_val_dice,
    hu130_val_dice,
):
    return {
        "epoch": epoch,
        "model_state_dict": (
            model.state_dict()
        ),
        "optimizer_state_dict": (
            optimizer.state_dict()
        ),
        "scheduler_state_dict": (
            scheduler.state_dict()
        ),
        "scaler_state_dict": (
            scaler.state_dict()
        ),
        "train_loss": train_loss,
        "raw_val_dice": (
            raw_val_dice
        ),
        "hu130_val_dice": (
            hu130_val_dice
        ),
        "best_metric": best_metric,
        "best_metric_epoch": (
            best_metric_epoch
        ),
        "foreground_ce_weight": (
            FOREGROUND_CE_WEIGHT
        ),
        "tversky_alpha": 0.3,
        "tversky_beta": 0.7,
        "crop_y": (36, 436),
        "crop_x": (62, 462),
        "window_depth": (
            WINDOW_DEPTH
        ),
        "window_stride": (
            WINDOW_STRIDE
        ),
        "hu_min": HU_MIN,
        "hu_max": HU_MAX,
        "num_classes": 2,
    }


# 현재 epoch 1 모델을 최초 best로 보존
if not best_model_path.exists():
    epoch1_row = (
        history_dataframe[
            history_dataframe[
                "epoch"
            ].astype(int) == 1
        ].iloc[0]
    )

    epoch1_checkpoint = (
        make_checkpoint(
            epoch=1,
            train_loss=float(
                epoch1_row[
                    "train_loss"
                ]
            ),
            raw_val_dice=float(
                epoch1_row[
                    "raw_val_dice"
                ]
            ),
            hu130_val_dice=float(
                epoch1_row[
                    "hu130_val_dice"
                ]
            ),
        )
    )

    torch.save(
        epoch1_checkpoint,
        best_model_path,
    )

    print(
        "epoch 1 모델을 최초 best로 저장"
    )


full_training_start = (
    time.perf_counter()
)


epoch_progress = tqdm(
    range(
        START_EPOCH,
        FINAL_EPOCH + 1,
    ),
    desc="전체 학습",
    unit="epoch",
)


for epoch in epoch_progress:
    epoch_start = (
        time.perf_counter()
    )


    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


    # 1. 학습
    model.train()
    epoch_loss_sum = 0.0


    batch_progress = tqdm(
        train_loader,
        total=len(train_loader),
        desc=(
            f"Epoch {epoch}/{FINAL_EPOCH}"
        ),
        unit="batch",
        leave=False,
    )


    for step, batch_data in enumerate(
        batch_progress,
        start=1,
    ):
        images = batch_data[
            "image"
        ].to(
            device,
            non_blocking=True,
        )

        labels = batch_data[
            "label"
        ].to(
            device,
            non_blocking=True,
        )


        optimizer.zero_grad(
            set_to_none=True,
        )


        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(
                device.type == "cuda"
            ),
        ):
            outputs = model(images)

            loss = loss_function(
                outputs,
                labels.unsqueeze(1),
            )


        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Epoch {epoch}, "
                f"batch {step}: "
                f"Loss가 유효하지 않습니다."
            )


        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()


        loss_value = float(
            loss.detach()
        )

        epoch_loss_sum += loss_value

        average_running_loss = (
            epoch_loss_sum / step
        )


        batch_progress.set_postfix(
            loss=f"{loss_value:.4f}",
            average=(
                f"{average_running_loss:.4f}"
            ),
        )


    average_train_loss = (
        epoch_loss_sum
        / len(train_loader)
    )


    # 2. 학습률 갱신
    scheduler.step()


    # 3. 매 epoch 환자 단위 검증
    validation_summary, (
        validation_results
    ) = validate_model(
        max_patients=None,
        show_progress=True,
    )


    raw_dice = float(
        validation_summary[
            "raw_dice"
        ]
    )

    hu130_dice = float(
        validation_summary[
            "hu130_dice"
        ]
    )


    # 4. 최적 모델 판정
    is_best = (
        hu130_dice > best_metric
    )

    if is_best:
        best_metric = hu130_dice
        best_metric_epoch = epoch


    # 5. 기록 추가
    current_learning_rate = float(
        optimizer.param_groups[0]["lr"]
    )

    epoch_minutes = (
        time.perf_counter()
        - epoch_start
    ) / 60


    training_history.append(
        {
            "epoch": epoch,
            "train_loss": (
                average_train_loss
            ),
            "raw_val_dice": (
                raw_dice
            ),
            "hu130_val_dice": (
                hu130_dice
            ),
            "raw_precision": (
                validation_summary[
                    "raw_precision"
                ]
            ),
            "raw_recall": (
                validation_summary[
                    "raw_recall"
                ]
            ),
            "hu130_precision": (
                validation_summary[
                    "hu130_precision"
                ]
            ),
            "hu130_recall": (
                validation_summary[
                    "hu130_recall"
                ]
            ),
            "learning_rate": (
                current_learning_rate
            ),
            "epoch_minutes": (
                epoch_minutes
            ),
        }
    )


    checkpoint_data = (
        make_checkpoint(
            epoch=epoch,
            train_loss=(
                average_train_loss
            ),
            raw_val_dice=raw_dice,
            hu130_val_dice=(
                hu130_dice
            ),
        )
    )


    # 6. 마지막 완료 모델 저장
    torch.save(
        checkpoint_data,
        last_model_path,
    )


    # 7. 최적 모델 저장
    if is_best:
        torch.save(
            checkpoint_data,
            best_model_path,
        )


    # 8. 환자별 검증 결과 저장
    validation_results.to_csv(
        RESULT_DIR
        / (
            f"validation_"
            f"epoch{epoch}.csv"
        ),
        index=False,
    )


    # 9. 전체 학습 기록 저장
    pd.DataFrame(
        training_history
    ).drop_duplicates(
        subset=["epoch"],
        keep="last",
    ).sort_values(
        "epoch"
    ).to_csv(
        history_path,
        index=False,
    )


    max_gpu_memory = 0.0

    if device.type == "cuda":
        max_gpu_memory = (
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        )


    # 10. Epoch 결과 출력
    print(
        f"\nEpoch {epoch}/{FINAL_EPOCH} 완료"
    )

    print(
        f"- 평균 Train Loss: "
        f"{average_train_loss:.4f}"
    )

    print(
        f"- 기본 Dice: "
        f"{raw_dice:.4f}"
    )

    print(
        f"- 기본 Precision: "
        f"{validation_summary['raw_precision']:.4f}"
    )

    print(
        f"- 기본 Recall: "
        f"{validation_summary['raw_recall']:.4f}"
    )

    print(
        f"- HU 130 Dice: "
        f"{hu130_dice:.4f}"
    )

    print(
        f"- HU 130 Precision: "
        f"{validation_summary['hu130_precision']:.4f}"
    )

    print(
        f"- HU 130 Recall: "
        f"{validation_summary['hu130_recall']:.4f}"
    )

    print(
        f"- 학습률: "
        f"{current_learning_rate:.8f}"
    )

    print(
        f"- Epoch 소요 시간: "
        f"{epoch_minutes:.1f}분"
    )

    print(
        f"- 최대 GPU 메모리: "
        f"{max_gpu_memory:.2f}GB"
    )

    print(
        f"- 현재 최고 HU 130 Dice: "
        f"{best_metric:.4f} "
        f"(Epoch {best_metric_epoch})"
    )

    if is_best:
        print(
            "- 새로운 최적 모델 저장 완료"
        )


    epoch_progress.set_postfix(
        loss=f"{average_train_loss:.4f}",
        raw=f"{raw_dice:.4f}",
        hu130=f"{hu130_dice:.4f}",
        best=f"{best_metric:.4f}",
    )


total_training_hours = (
    time.perf_counter()
    - full_training_start
) / 3600


print(
    f"\n{'=' * 50}"
)
print("Epoch 1~20 학습 완료")
print(
    f"{'=' * 50}"
)

print(
    "최고 HU 130 Dice:",
    round(best_metric, 4),
)

print(
    "최고 성능 Epoch:",
    best_metric_epoch,
)

print(
    "추가 학습 시간:",
    round(
        total_training_hours,
        2,
    ),
    "시간",
)

print(
    "최적 모델:",
    best_model_path,
)

print(
    "마지막 모델:",
    last_model_path,
)

print(
    "학습 기록:",
    history_path,
)

학습 시작 epoch: 2
최종 epoch: 20
현재 최고 HU 130 Dice: 0.0012851407224768
현재 최고 epoch: 1
epoch 1 모델을 최초 best로 저장


전체 학습:   0%|          | 0/19 [00:00<?, ?epoch/s]

Epoch 2/20:   0%|          | 0/688 [00:00<?, ?batch/s]

환자 단위 검증:   0%|          | 0/43 [00:00<?, ?it/s]


Epoch 2/20 완료
- 평균 Train Loss: 1.1509
- 기본 Dice: 0.1106
- 기본 Precision: 0.0672
- 기본 Recall: 0.5232
- HU 130 Dice: 0.1818
- HU 130 Precision: 0.1336
- HU 130 Recall: 0.5232
- 학습률: 0.00019922
- Epoch 소요 시간: 13.5분
- 최대 GPU 메모리: 4.15GB
- 현재 최고 HU 130 Dice: 0.1818 (Epoch 2)
- 새로운 최적 모델 저장 완료


...

환자 단위 검증:   0%|          | 0/43 [00:00<?, ?it/s]


Epoch 19/20 완료
- 평균 Train Loss: 0.1879
- 기본 Dice: 0.8199
- 기본 Precision: 0.7442
- 기본 Recall: 0.9497
- HU 130 Dice: 0.9392
- HU 130 Precision: 0.9418
- HU 130 Recall: 0.9497
- 학습률: 0.00013713
- Epoch 소요 시간: 13.3분
- 최대 GPU 메모리: 4.15GB
- 현재 최고 HU 130 Dice: 0.9392 (Epoch 19)
- 새로운 최적 모델 저장 완료


Epoch 20/20:   0%|          | 0/688 [00:00<?, ?batch/s]

환자 단위 검증:   0%|          | 0/43 [00:00<?, ?it/s]


Epoch 20/20 완료
- 평균 Train Loss: 0.1854
- 기본 Dice: 0.8084
- 기본 Precision: 0.7283
- 기본 Recall: 0.9664
- HU 130 Dice: 0.8758
- HU 130 Precision: 0.8407
- HU 130 Recall: 0.9664
- 학습률: 0.00013125
- Epoch 소요 시간: 13.5분
- 최대 GPU 메모리: 4.15GB
- 현재 최고 HU 130 Dice: 0.9392 (Epoch 19)

Epoch 1~20 학습 완료
최고 HU 130 Dice: 0.9392
최고 성능 Epoch: 19
추가 학습 시간: 4.42 시간
최적 모델: /content/drive/MyDrive/COCA_COMMON_BINARY/segresnet_results/segresnet_best_hu130.pth
마지막 모델: /content/drive/MyDrive/COCA_COMMON_BINARY/segresnet_results/segresnet_last.pth
학습 기록: /content/drive/MyDrive/COCA_COMMON_BINARY/segresnet_results/training_history_tversky_weighted.csv
